# Setup

In [ ]:
# Run ONLY once. Working directory should be 02-machine-translation
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", Path.cwd())

Working directory: /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026/02-machine-translation


In [97]:
# Load autoreload extension for Jupyter Notebook
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
from src.machine_translation.config import TatoebaConfig
from src.machine_translation.dataset import TatoebaData
from src.machine_translation.model import TatoebaModelPackedSeq

# Tell PyTorch it is safe to load your custom Config class
import torch
torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])
import torch.nn.functional as F
import torch.nn as nn

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Set up a logger for "tatoeba" level to DEBUG for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.DEBUG)

In [35]:
# Set up a logger for "tatoeba" level to INFO for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.INFO)

# 01 `TatoebaData` class

## 01 Loading the dataset

In [7]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config)

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from datasets/tatoeba...
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157839, validation=39460, test=24514


In [6]:
# Inspect datasets or fetch a batch from a dataloader
print("Train samples:", len(data.train_data))
print("Sample:", data.train_data[0])

Train samples: 157839
Sample: {'source_text': 'Tom tried to break up the fight.', 'target_text': 'Tom trató de disolver la pelea.', 'source_lang': 'eng', 'target_lang': 'spa'}


In [ ]:
# Inspect a batch from DataLoader BEFORE tokenization
train_loader = data.train_dataloader()
batch = next(iter(train_loader))
print("Batch type:", type(batch), "Batch keys:", batch.keys())
print("Batch size:", len(batch['source_text']), len(batch['target_text']))
print("Sample source text:", batch['source_text'][0])

Batch type: <class 'dict'> Batch keys: dict_keys(['source_text', 'target_text', 'source_lang', 'target_lang'])
Batch size: 32 32
Sample source text: He can't be ill.


## 02 Training a BPE tokenizer from scratch

In [28]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config)

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from datasets/tatoeba...
DEBUG:tatoeba.dataset:  Training BPE tokenizer with vocab_size=10000 and max_seq_length=256...


DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=10000, pad_id=0
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157839, validation=39460, test=24514


In [29]:
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")
print(f"<pad> ID: {tokenizer.token_to_id('<pad>')}")
print(f"<unk> ID: {tokenizer.token_to_id('<unk>')}")
print(f"<s> ID: {tokenizer.token_to_id('<s>')}")
print(f"</s> ID: {tokenizer.token_to_id('</s>')}")

Vocabulary size: 10000
<pad> ID: 0
<unk> ID: 1
<s> ID: 2
</s> ID: 3


In [30]:
# Inspect one English/Spanish pair and its encoded result
example = data.train_data[0]

source_text = example["source_text"]
target_text = example["target_text"]

source_encoding = tokenizer.encode(source_text)
target_encoding = tokenizer.encode(target_text)

print("English:", source_text)
print("English tokens:", source_encoding.tokens)
print("English IDs:", source_encoding.ids)
print()
print("Spanish:", target_text)
print("Spanish tokens:", target_encoding.tokens)
print("Spanish IDs:", target_encoding.ids)

English: Tom tried to break up the fight.
English tokens: ['Tom', 'tried', 'to', 'break', 'up', 'the', 'fight', '.']
English IDs: [217, 1826, 196, 1600, 444, 214, 3006, 17]

Spanish: Tom trató de disolver la pelea.
Spanish tokens: ['Tom', 'trató', 'de', 'dis', 'olver', 'la', 'pelea', '.']
Spanish IDs: [217, 4654, 201, 462, 3069, 210, 7759, 17]


In [31]:
# Confirm padding and truncation work
encodings = tokenizer.encode_batch([
    data.train_data[0]["source_text"],
    data.train_data[1]["source_text"],
])

for encoding in encodings:
    print("length:", len(encoding.ids))
    print("ids:", encoding.ids)
    print("attention mask:", encoding.attention_mask)

length: 8
ids: [217, 1826, 196, 1600, 444, 214, 3006, 17]
attention mask: [1, 1, 1, 1, 1, 1, 1, 1]
length: 8
ids: [4522, 43, 315, 9384, 17, 0, 0, 0]
attention mask: [1, 1, 1, 1, 1, 0, 0, 0]


In [32]:
type(encodings), len(encodings), type(encodings[0])

(list, 2, tokenizers.Encoding)

In [33]:
encodings[0]

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

## 03 Data Limit for Quick Experiments

In [4]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=None)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Local cache not found at datasets/tatoeba. Downloading Tatoeba dataset from Hugging Face...


Generating validation split:   0%|          | 0/197299 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24514 [00:00<?, ? examples/s]

DEBUG:tatoeba.dataset:  Training BPE tokenizer with max_vocab_size=10000 and max_seq_length=256...


DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=10000, pad_id=0
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157,839, validation=39,460, test=24,514
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer.json.
DEBUG:tatoeba.dataset:  Vocab size: 10,000


In [10]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=None)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 157,839
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157,839, validation=39,460, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer.json.
DEBUG:tatoeba.dataset:  Vocab size: 10,000


In [11]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  === _train_tokenizer() call ===
DEBUG:tatoeba.dataset:  Training BPE tokenizer with max_vocab_size=10,000 and max_seq_length=256...
DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=5,652, pad_id=0
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [12]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


## 04 Custom `collate_fn` for the DataLoader

In [13]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [19]:
data.train_data[0]

{'source_text': 'None of us are opposed to his ideas.',
 'target_text': 'Ninguno de nosotros está en contra de sus ideas.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [23]:
# Create train dataloader for the training dataset
train_dataloader = data.train_dataloader()

# Fetch a batch from the dataloader and inspect it
batch = next(iter(train_dataloader))

DEBUG:tatoeba.dataset:  === _collate_fn() call ===
DEBUG:tatoeba.dataset:  Batch type: <class 'list'>
DEBUG:tatoeba.dataset:  Batch keys: ['source_text', 'target_text', 'source_lang', 'target_lang']
DEBUG:tatoeba.dataset:  Batch example source_text: We should have gotten married.
DEBUG:tatoeba.dataset:  Batch example target_text: Deberíamos habernos casado.
DEBUG:tatoeba.dataset:  Batch source_texts length: 32
DEBUG:tatoeba.dataset:  Batch example source_text: We should have gotten married.
DEBUG:tatoeba.dataset:  Batch target_texts length: 32
DEBUG:tatoeba.dataset:  Batch example target_text: <s> Deberíamos habernos casado. </s>
DEBUG:tatoeba.dataset:  Source encodings length: 32
DEBUG:tatoeba.dataset:  Source encoding example encodings: Encoding(num_tokens=17, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
DEBUG:tatoeba.dataset:  Source encoding example ids: [286, 495, 217, 4302, 2174, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
DEBUG:tatoeba.

# 02 `TatoebaModelPackedSeq` class

## 01 `__init__` method

In [6]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [8]:
# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config, data)

DEBUG:tatoeba.model:  ===MODEL CREATION===
DEBUG:tatoeba.model:  Model initialized with vocab_size: 5652:,
DEBUG:tatoeba.model:  Model hyperparameters: hidden_dim=512, num_layers=2, embedding_dim=512, dropout=0.0
DEBUG:tatoeba.model:  Embedding layer created with embedding_dim: 512
DEBUG:tatoeba.model:  Embedding layer dimensions: torch.Size([5652, 512])
DEBUG:tatoeba.model:  Encoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Decoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  LayerNorm created with hidden_dim: 512
DEBUG:tatoeba.model:  Linear layer created with input_dim: 512, output_dim: 5,652
DEBUG:tatoeba.model:  Custom weight initialization...


## 02 `forward` method

In [8]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

# Get one batch from the dataloader
batch = next(iter(data.train_dataloader()))
encoder_input_ids = batch["encoder_input_ids"]
encoder_attention_mask = batch["encoder_attention_mask"]
decoder_input_ids = batch["decoder_input_ids"]

DEBUG:tatoeba.dataset:  === DATASET CREATION ===
DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652
DEBUG:tatoeba.dataset:  === _collate

In [7]:
# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config, data)

DEBUG:tatoeba.model:  ===MODEL CREATION===
DEBUG:tatoeba.model:  Model initialized with vocab_size: 5652:,
DEBUG:tatoeba.model:  Model hyperparameters: hidden_dim=512, num_layers=2, embedding_dim=512, dropout=0.0
DEBUG:tatoeba.model:  Embedding layer created with embedding_dim: 512
DEBUG:tatoeba.model:  Embedding layer dimensions: torch.Size([5652, 512])
DEBUG:tatoeba.model:  Encoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Decoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  LayerNorm created with hidden_dim: 512
DEBUG:tatoeba.model:  Linear layer created with input_dim: 512, output_dim: 5,652
DEBUG:tatoeba.model:  Custom weight initialization...


In [9]:
# Test the forward pass
logits = model(encoder_input_ids, encoder_attention_mask, decoder_input_ids)

DEBUG:tatoeba.model:  === forward() call ===
DEBUG:tatoeba.model:  ---Input shapes---
DEBUG:tatoeba.model:  Encoder input_ids shape: torch.Size([32, 22])
DEBUG:tatoeba.model:  Encoder attention_mask shape: torch.Size([32, 22])
DEBUG:tatoeba.model:  Decoder input_ids shape: torch.Size([32, 20])
DEBUG:tatoeba.model:  ---Embeddings shapes---
DEBUG:tatoeba.model:  (1) Embeddings output shape: torch.Size([32, 22, 512])
DEBUG:tatoeba.model:  (1) Decoder embeddings output shape: torch.Size([32, 20, 512])
DEBUG:tatoeba.model:  ---Packing sequences---
DEBUG:tatoeba.model:  (2) Attention mask shape: torch.Size([32, 22])
DEBUG:tatoeba.model:  (2) Lengths tensor: tensor([ 5,  6,  5, 22,  9])
DEBUG:tatoeba.model:  ---GRU outputs---
DEBUG:tatoeba.model:  (3) Hidden dimension: 512
DEBUG:tatoeba.model:  (3) Encoder output shape: torch.Size([300, 512]) Type: <class 'torch.nn.utils.rnn.PackedSequence'>
DEBUG:tatoeba.model:  (3) Encoder last hidden state shape: torch.Size([2, 32, 512]) Type: <class 'torc

## 03 `__init__` method with changes

In [10]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === DATASET CREATION ===
DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [11]:
# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config=config, pad_id=data.pad_id, vocab_size=data.vocab_size)

DEBUG:tatoeba.model:  ===MODEL CREATION===
DEBUG:tatoeba.model:  Model initialized with vocab_size: 5,652, pad_id: 0
DEBUG:tatoeba.model:  Model hyperparameters: hidden_dim=512, num_layers=2, embedding_dim=512, dropout=0.0
DEBUG:tatoeba.model:  Embedding layer created with embedding_dim: 512
DEBUG:tatoeba.model:  Embedding layer dimensions: torch.Size([5652, 512])
DEBUG:tatoeba.model:  Encoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Decoder created with input_size: 512, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  LayerNorm created with hidden_dim: 512
DEBUG:tatoeba.model:  Linear layer created with input_dim: 512, output_dim: 5,652
DEBUG:tatoeba.model:  Custom weight initialization...


## 04 Computing the speed of decoding

In [15]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()
config.batch_size = 2

print(f"Batch size: {config.batch_size}")

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

# Get one batch from the dataloader
batch = next(iter(data.train_dataloader()))
encoder_input_ids = batch["encoder_input_ids"]
encoder_attention_mask = batch["encoder_attention_mask"]
decoder_input_ids = batch["decoder_input_ids"]

Batch size: 2


In [17]:
type(batch), batch.keys(), len(batch["encoder_input_ids"])

(dict,
 dict_keys(['encoder_input_ids', 'encoder_attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'decoder_labels']),
 2)

In [18]:
batch["encoder_input_ids"]

tensor([[ 230,    7,   64,  199,  985,   10,    0],
        [ 286,  322,   46, 3869,  144, 4153,   10]])

In [19]:
batch["encoder_attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1]])

In [21]:
batch["decoder_input_ids"]

tensor([[   2,  141,  193, 4205,   10,    3,    0],
        [   2, 5619,  174, 3871,   98, 4151,   10]])

In [20]:
batch["decoder_labels"]

tensor([[ 141,  193, 4205,   10,    3,    0,    0],
        [5619,  174, 3871,   98, 4151,   10,    3]])

Let's trace some sentences in validation data since they are not shuffled data.

In [27]:
batch_val = next(iter(data.val_dataloader()))

In [23]:
data.val_data[0], data.val_data[1]

({'source_text': "I can't walk, but I can definitely hobble.",
  'target_text': 'No puedo andar, pero me las apaño cojeando.',
  'source_lang': 'eng',
  'target_lang': 'spa'},
 {'source_text': 'When did life come into being?',
  'target_text': '¿Cuándo apareció la vida?',
  'source_lang': 'eng',
  'target_lang': 'spa'})

In [24]:
batch_collate = [data.val_data[0], data.val_data[1]]
src_texts = [entry['source_text'] for entry in batch_collate]
tgt_texts = [f"{data.BOS_TOKEN} {entry['target_text']} {data.EOS_TOKEN}" for entry in batch_collate]
src_encodings = data.tokenizer.encode_batch(src_texts)
tgt_encodings = data.tokenizer.encode_batch(tgt_texts)
# Extract input IDs and attention masks from the encodings and convert them to PyTorch tensors
src_ids = torch.tensor([enc.ids for enc in src_encodings], dtype=torch.long)
src_mask = torch.tensor([enc.attention_mask for enc in src_encodings], dtype=torch.long)
# Extract target IDs and attention masks from the encodings and convert them to PyTorch tensors
tgt_ids = torch.tensor([enc.ids for enc in tgt_encodings], dtype=torch.long)
tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings], dtype=torch.long)

batch_manual = {
            "encoder_input_ids": src_ids,
            "encoder_attention_mask": src_mask,
            "decoder_input_ids": tgt_ids[:, :-1],    # Drops </s> (Starts with <s>) [batch_size, seq_length-1]
            "decoder_attention_mask": tgt_mask[:, :-1], # 
            "decoder_labels": tgt_ids[:, 1:],                 # Drops <s> (Ends with </s>) [batch_size, seq_length-1]
        }

In [25]:
batch_manual

{'encoder_input_ids': tensor([[  29,  165,    7,   65, 1611,    8,  579,   29,  165,   98,  373, 1766,
           279,  189,   47,  239,   10],
         [ 980,  275, 1424,  386, 1880, 1127,   20,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0]]),
 'encoder_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
 'decoder_input_ids': tensor([[   2,  146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,
           304,  310,   10],
         [   2,   73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,
             0,    0,    0]]),
 'decoder_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]]),
 'decoder_labels': tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
           310,   10,    3],
         [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,  

In [28]:
batch_val

{'encoder_input_ids': tensor([[  29,  165,    7,   65, 1611,    8,  579,   29,  165,   98,  373, 1766,
           279,  189,   47,  239,   10],
         [ 980,  275, 1424,  386, 1880, 1127,   20,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0]]),
 'encoder_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
 'decoder_input_ids': tensor([[   2,  146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,
           304,  310,   10],
         [   2,   73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,
             0,    0,    0]]),
 'decoder_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]]),
 'decoder_labels': tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
           310,   10,    3],
         [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,  

Let's now look at how we shift the target sequences for the decoder input and labels in the `_collate_fn` method. this is called "teacher forcing."

In [30]:
tgt_ids


tensor([[   2,  146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,
          304,  310,   10,    3],
        [   2,   73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,
            0,    0,    0,    0]])

In [32]:
# Target Input: Removed the last token for teacher forcing in the decoder input
tgt_ids[:, :-1]

tensor([[   2,  146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,
          304,  310,   10],
        [   2,   73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,
            0,    0,    0]])

In [33]:
# Target Labels: Removed the first token for the labels in the decoder output
tgt_ids[:, 1:]

tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
          310,   10,    3],
        [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,    0,
            0,    0,    0]])

Finally, we may compute the speed of our model by measuring the number of target tokens processed per second.

In [36]:
batch_val["decoder_labels"]

tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
          310,   10,    3],
        [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,    0,
            0,    0,    0]])

In [37]:
len(batch_val["decoder_labels"][0]) + len(batch_val["decoder_labels"][1]) - 5

25

In [38]:
batch_val["decoder_labels"] != data.pad_id

tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         False, False, False, False, False]])

In [35]:
target_tokens = (batch_val["decoder_labels"] != data.pad_id).sum()
target_tokens

tensor(25)

## 05 `__init__` Method (2)

In [10]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()
config.embedding_dim = 256

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === DATASET CREATION ===
DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [11]:
# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config, tokenizer=data.tokenizer)

DEBUG:tatoeba.model:  ===MODEL CREATION===
DEBUG:tatoeba.model:  Model initialized with vocab_size: 5,652, pad_id: 0
DEBUG:tatoeba.model:  Model hyperparameters: hidden_dim=512, num_layers=2, embedding_dim=256, dropout=0.0
DEBUG:tatoeba.model:  Embedding layer dimensions: torch.Size([5652, 256])
DEBUG:tatoeba.model:  Encoder created with input_size: 256, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Decoder created with input_size: 256, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Linear layer created with input_dim: 512, output_dim: 5,652


## 06 `forward` Method (2)

In [ ]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()
config.batch_size = 2
config.embedding_dim = 256

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

# Get one batch from the dataloader
batch = next(iter(data.train_dataloader()))
encoder_input_ids = batch["encoder_input_ids"]
encoder_attention_mask = batch["encoder_attention_mask"]
decoder_input_ids = batch["decoder_input_ids"]
decoder_labels = batch["decoder_labels"]

torch.Size([2, 12])


In [13]:
# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config, tokenizer=data.tokenizer)

DEBUG:tatoeba.model:  ===MODEL CREATION===
DEBUG:tatoeba.model:  Model initialized with vocab_size: 5,652, pad_id: 0
DEBUG:tatoeba.model:  Model hyperparameters: hidden_dim=512, num_layers=2, embedding_dim=256, dropout=0.0
DEBUG:tatoeba.model:  Embedding layer dimensions: torch.Size([5652, 256])
DEBUG:tatoeba.model:  Encoder created with input_size: 256, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Decoder created with input_size: 256, hidden_size: 512, num_layers: 2, dropout: 0.0
DEBUG:tatoeba.model:  Linear layer created with input_dim: 512, output_dim: 5,652


In [21]:
# Test the forward pass
logits = model(encoder_input_ids, encoder_attention_mask, decoder_input_ids)

DEBUG:tatoeba.model:  === forward() call ===
DEBUG:tatoeba.model:  ---Input shapes---
DEBUG:tatoeba.model:  Encoder input_ids shape: torch.Size([2, 10])
DEBUG:tatoeba.model:  Encoder input_ids type: <class 'torch.Tensor'>
DEBUG:tatoeba.model:  Encode input ids: tensor([[ 354,  450, 1591,  219,   46, 3795, 2350,   10,    0,    0],
        [2338,  109,    7,  122,  490,  516,  117,  573,  272,    4]])
DEBUG:tatoeba.model:  Encoder attention_mask shape: torch.Size([2, 10])
DEBUG:tatoeba.model:  Decoder input_ids shape: torch.Size([2, 9])
DEBUG:tatoeba.model:  ---Embeddings shapes---
DEBUG:tatoeba.model:  (1) Encoder embeddings output shape: torch.Size([2, 10, 256])
DEBUG:tatoeba.model:  (1) Decoder embeddings output shape: torch.Size([2, 9, 256])
DEBUG:tatoeba.model:  ---Packing sequences---
DEBUG:tatoeba.model:  (2) Lengths tensor: tensor([ 8, 10])
DEBUG:tatoeba.model:  (2) Packed embeddings data shape: torch.Size([18, 256]) Type: <class 'torch.nn.utils.rnn.PackedSequence'>
DEBUG:tatoeba

In [25]:
config.batch_size, data.vocab_size, decoder_input_ids.shape

(2, 5652, torch.Size([2, 9]))

In [ ]:
logits.shape # [batch_size, vocab_size, seq_length]

torch.Size([2, 5652, 9])

In [27]:
logits[:, 0, :]

tensor([[ 0.1178,  0.6675,  0.1835,  0.7548,  1.1802,  0.9619,  1.6802,  2.0784,
          2.3058],
        [-1.1219,  0.7600,  1.1385,  0.6811,  0.8845,  0.5067,  0.9998,  1.2949,
          1.4798]], grad_fn=<SelectBackward0>)

In [29]:
decoder_labels

tensor([[4862, 5485,  380, 3562,  120,  102, 2088,   10,    3,    0,    0,    0],
        [ 718, 1572,   46, 2486,    8,  900,  116,  123, 2138, 4220,   10,    3]])

## 07 `CrossEntropyLoss`

In [56]:
# Set seed for reproducibility
torch.manual_seed(42)

# Create a congig object for Tatoeba dataset
config = TatoebaConfig()
config.batch_size = 2
config.embedding_dim = 256

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

# Get one batch from the dataloader
batch = next(iter(data.val_dataloader()))
encoder_input_ids = batch["encoder_input_ids"]
encoder_attention_mask = batch["encoder_attention_mask"]
decoder_input_ids = batch["decoder_input_ids"]
decoder_labels = batch["decoder_labels"]
print(f"decoder_labels shape: {decoder_labels.shape}")
print(f"decoder_labels\n{decoder_labels}")

# Create an instance of the TatoebaModelPackedSeq using the config and data
model = TatoebaModelPackedSeq(config, tokenizer=data.tokenizer)

# Test the forward pass
logits = model(encoder_input_ids, encoder_attention_mask, decoder_input_ids)

# Compute the loss
loss = model.loss_fn(logits, decoder_labels)
print(f"CrossEntropyLoss: {loss.item()}")

decoder_labels shape: torch.Size([2, 15])
decoder_labels
tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
          310,   10,    3],
        [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,    0,
            0,    0,    0]])
CrossEntropyLoss: 9.65877914428711


In [96]:
# Extract the padding token ID from the data object
pad_id = data.pad_id

# Built-in result
loss_builtin = model.loss_fn(logits, decoder_labels)

# Log probabilities over the vocabulary dimension
log_probs = F.log_softmax(logits, dim=1)
print(f"logits shape: {logits.shape}")
print(f"log_probs shape: {log_probs.shape}")

# Select the log probability assigned to each correct target token.
# decoder_labels has shape [batch_size, sequence_length].
# unsqueeze(1) makes it [batch_size, 1, sequence_length],
# matching logits' vocabulary dimension.
target_log_probs = torch.gather(
    log_probs,
    dim=1,
    index=decoder_labels.unsqueeze(1),
).squeeze(1)

print(f"decoder_labels shape: {decoder_labels.shape}")
print(f"target_log_probs shape: {target_log_probs.shape}")

# Negative log likelihood for each target position
token_losses = -target_log_probs

# Keep only non-padding positions
print(f"\ndecoder_labels:\n{decoder_labels}")
valid_positions = decoder_labels != pad_id
print(f"valid_positions:\n{valid_positions}")

print(f"\ntoken_losses:\n{token_losses}")
valid_token_losses = token_losses[valid_positions]
print(f"valid_token_losses shape: {valid_token_losses.shape}")
print(f"valid_token_losses:\n{valid_token_losses}")

# Mean over valid tokens, matching CrossEntropyLoss's default reduction="mean"
loss_manual = valid_token_losses.mean()

print(f"\n\nBuilt-in loss: {loss_builtin.item()}")
print(f"Manual loss:   {loss_manual.item()}")
print(f"Valid tokens:  {valid_positions.sum().item()}")
print(f"Match:         {torch.allclose(loss_builtin, loss_manual)}")

logits shape: torch.Size([2, 5652, 15])
log_probs shape: torch.Size([2, 5652, 15])
decoder_labels shape: torch.Size([2, 15])
target_log_probs shape: torch.Size([2, 15])

decoder_labels:
tensor([[ 146,  453,  196,   87,    8,  492,  117,  234, 1696,  683,  140,  304,
          310,   10,    3],
        [  73, 1221,   46,  168,   50, 1412,  107, 1133,   20,    3,    0,    0,
            0,    0,    0]])
valid_positions:
tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         False, False, False, False, False]])

token_losses:
tensor([[ 8.5975, 10.8099, 11.0036, 11.8186,  8.6879, 10.5734,  9.6083,  9.2613,
          8.9943,  7.1947,  9.3705,  9.0263, 11.3649, 12.5686,  8.8239],
        [ 8.4826,  8.5163,  9.4691, 10.0865, 10.2778,  9.6147,  8.5417,  9.2792,
         10.9491,  8.5487, 10.0626, 10.3787, 10.3400, 10.0641,  9.6763]],
     

### 01 `F.log_softmax`

In [61]:
torch.manual_seed(42)
input = torch.randn(2, 3)
print(f"{input}")

tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])


In [63]:
softmax = F.softmax(input, dim=1)

In [64]:
softmax

tensor([[0.3683, 0.2992, 0.3325],
        [0.5215, 0.1348, 0.3438]])

In [65]:
softmax.sum(dim=1)

tensor([1., 1.])

In [67]:
torch.log(softmax)

tensor([[-0.9988, -1.2067, -1.1011],
        [-0.6511, -2.0043, -1.0678]])

In [68]:
F.log_softmax(input, dim=1)

tensor([[-0.9988, -1.2067, -1.1011],
        [-0.6511, -2.0043, -1.0678]])

### 02 `torch.gather`

In [69]:
values = torch.tensor([[10, 20, 30],
                       [40, 50, 60]])

indices = torch.tensor([[2],
                        [0]])

torch.gather(values, dim=1, index=indices)

tensor([[30],
        [40]])

In [71]:
values = torch.tensor([[10, 20, 30],
                       [40, 50, 60]])

indices = torch.tensor([[2, 1],
                        [0, 1]])

torch.gather(values, dim=1, index=indices)

tensor([[30, 20],
        [40, 50]])

In [72]:
target_log_probs = torch.gather(
    log_probs,
    dim=1,
    index=decoder_labels.unsqueeze(1),
).squeeze(1)

In [75]:
log_probs.shape, decoder_labels.shape, decoder_labels.unsqueeze(1).shape

(torch.Size([2, 5652, 15]), torch.Size([2, 15]), torch.Size([2, 1, 15]))

In [79]:
log_probs_v1 = torch.tensor([
    # Batch item 0
    [
        [-1.0, -2.0, -3.0],  
        [-4.0, -5.0, -6.0],  
        [-7.0, -8.0, -9.0],  
        [-10.0, -11.0, -12.0],  
    ],

    # Batch item 1
    [
        [-13.0, -14.0, -15.0],  
        [-16.0, -17.0, -18.0],  
        [-19.0, -20.0, -21.0],  
        [-22.0, -23.0, -24.0],  
    ],
])

decoder_labels_v1 = torch.tensor([
    [2, 1, 0],  
    [3, 0, 2],  
])

target_log_probs_v1 = torch.gather(
    log_probs_v1,
    dim=1,
    index=decoder_labels_v1.unsqueeze(1),
)

In [80]:
target_log_probs_v1

tensor([[[ -7.,  -5.,  -3.]],

        [[-22., -14., -21.]]])

In [81]:
target_log_probs_v1.shape

torch.Size([2, 1, 3])

In [83]:
target_log_probs_v1.squeeze(1)

tensor([[ -7.,  -5.,  -3.],
        [-22., -14., -21.]])

In [85]:
target_log_probs_v1.squeeze(1).shape

torch.Size([2, 3])

### 03 A Simple Loss Calculation

In [98]:
# --- SETUP ---
# Vocabulary size: 4 words [0: "cat", 1: "dog", 2: "fish", 3: "bird"]
# Suppose the true target word is "dog" (index 1)
target_index = 1
vocab_size = 4

# Raw unnormalized model outputs (logits)
logits = torch.tensor([2.0, 1.0, 0.1, -0.5])

In [100]:
# Step 1: Softmax to get probabilities q
probs = torch.exp(logits) / torch.exp(logits).sum()
print(f"probs:\n{probs}")

probs2 = F.softmax(logits, dim=0)
print(f"probs2:\n{probs2}")

probs:
tensor([0.6252, 0.2300, 0.0935, 0.0513])
probs2:
tensor([0.6252, 0.2300, 0.0935, 0.0513])


In [110]:
# Step 2: One-hot encoded vector p for target "dog" (index 1)
one_hot = torch.tensor([0.0, 1.0, 0.0, 0.0])

# Step 3: Manual cross-entropy loss calculation
loss_manual = -(one_hot * torch.log(probs)).sum()
print(f"Manual cross-entropy loss: {loss_manual}")

# Using bulean masks
mask = torch.zeros_like(probs, dtype=torch.bool)
mask[target_index] = True
loss_mask = -torch.log(probs[mask])
print(f"Cross-entropy loss using boolean mask: {loss_mask.item()}")

# Using LogSoftmax
log_probs = torch.log_softmax(logits, dim=0)
loss_log_softmax = -log_probs[target_index]
print(f"Cross-entropy loss using LogSoftmax: {loss_log_softmax.item()}")

Manual cross-entropy loss: 1.4697118997573853
Cross-entropy loss using boolean mask: 1.4697118997573853
Cross-entropy loss using LogSoftmax: 1.4697117805480957


In [109]:
import math
loss_math = -math.log(probs[target_index])
print(f"Cross-entropy loss using math.log: {loss_math}")

Cross-entropy loss using math.log: 1.469711844924073


In [112]:
# -------------------------------------------------------------
# METHOD 2: Using PyTorch's Built-in Loss
# -------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

# PyTorch expects logits shape (batch_size, num_classes) and target shape (batch_size)
pytorch_cross_entropy = criterion(logits.unsqueeze(0), torch.tensor([target_index]))
print(f"Cross-entropy loss using PyTorch's built-in loss: {pytorch_cross_entropy.item()}")

Cross-entropy loss using PyTorch's built-in loss: 1.4697117805480957
